In [ ]:
# bronze_btc_transform.py

# Import libraries
import json                     # For reading/writing JSON files
from datetime import datetime  # For converting timestamps to readable dates
import sys                     # For exiting program on error


# Function to load raw JSON file
def load_raw_json(filename):
    try:
        # Open the raw JSON file in read mode
        with open(filename, "r") as file:
            return json.load(file)  # Convert JSON → Python object (list)

    except IOError as e:
        # Handle file errors (missing file, wrong path, etc.)
        print(f"[ERROR] Failed to read file: {e}")
        sys.exit(1)


# Function to transform raw data into bronze format
def transform_to_bronze(data):
    formatted_data = []  # Will store cleaned records

    # Loop through each row in raw data
    # Each row looks like:
    # [timestamp, open, high, low, close, volume, ...]
    for row in data:
        try:
            # Convert raw list into structured dictionary
            record = {
                # Convert timestamp (milliseconds → seconds → readable date)
                "date": datetime.utcfromtimestamp(row[0] / 1000).strftime("%Y-%m-%d"),

                # Convert price/volume values to floats
                "open": float(row[1]),
                "high": float(row[2]),
                "low": float(row[3]),
                "close": float(row[4]),
                "volume": float(row[5]),
            }

            # Add cleaned record to list
            formatted_data.append(record)

        except (ValueError, IndexError) as e:
            # Skip rows with bad/missing data
            print(f"[WARNING] Skipping bad row: {row} | Error: {e}")

    # Remove duplicates using date as unique key
    # If duplicates exist, latest one overwrites previous
    unique_data = {item["date"]: item for item in formatted_data}

    # Convert back to list and sort by date (ascending)
    sorted_data = sorted(unique_data.values(), key=lambda x: x["date"])

    return sorted_data


# Function to save bronze data to JSON file
def save_bronze_json(data):
    # Create timestamped filename to avoid overwriting
    filename = f"btc_bronze_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

    try:
        # Open file in write mode
        with open(filename, "w") as file:
            # Save cleaned data with indentation (readable format)
            json.dump(data, file, indent=4)

        print(f"[SUCCESS] Bronze data saved to {filename}")

    except IOError as e:
        # Handle file writing errors
        print(f"[ERROR] Failed to write file: {e}")
        sys.exit(1)


# Main pipeline function
def run_pipeline():
    
    raw_file = "btc_raw_20260401_102624.json"

    print("[INFO] Loading raw data...")
    # Step 1: Load raw JSON file
    raw_data = load_raw_json(raw_file)

    print("[INFO] Transforming to bronze layer...")
    # Step 2: Clean + structure data
    bronze_data = transform_to_bronze(raw_data)

    print("[INFO] Saving bronze JSON...")
    # Step 3: Save transformed data
    save_bronze_json(bronze_data)


# Entry point (runs script)
if __name__ == "__main__":
    run_pipeline()

[INFO] Loading raw data...
[INFO] Transforming to bronze layer...
[INFO] Saving bronze JSON...
[SUCCESS] Bronze data saved to btc_bronze_20260401_103401.json


C:\Users\Christopher\AppData\Local\Temp\ipykernel_28264\1168004494.py:24: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "date": datetime.utcfromtimestamp(row[0] / 1000).strftime("%Y-%m-%d"),


In [6]:
# bronze_fng_transform.py

import json
from datetime import datetime
import sys


def load_raw_json(filename):
    try:
        with open(filename, "r") as f:
            return json.load(f)

    except IOError as e:
        print(f"[ERROR] Failed to read file: {e}")
        sys.exit(1)


def transform_to_bronze(data):
    formatted = []

    for row in data.get("data", []):
        try:
            record = {
                "date": datetime.utcfromtimestamp(int(row["timestamp"])).strftime("%Y-%m-%d"),
                "value": int(row["value"]),
                "classification": row["value_classification"]
            }

            formatted.append(record)

        except (KeyError, ValueError) as e:
            print(f"[WARNING] Skipping bad row: {row} | Error: {e}")

    # Remove duplicates by date
    unique_data = {item["date"]: item for item in formatted}

    # Sort by date
    sorted_data = sorted(unique_data.values(), key=lambda x: x["date"])

    return sorted_data


def save_bronze_json(data):
    filename = f"fng_bronze_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

    try:
        with open(filename, "w") as f:
            json.dump(data, f, indent=4)

        print(f"[SUCCESS] Bronze data saved to {filename}")

    except IOError as e:
        print(f"[ERROR] Failed to write file: {e}")
        sys.exit(1)


def run_pipeline():
    # ✅ INSERT YOUR FILENAME HERE
    raw_file = "fng_raw_20260401_104149.json"

    print("[INFO] Loading raw data...")
    raw_data = load_raw_json(raw_file)

    print("[INFO] Transforming to bronze layer...")
    bronze_data = transform_to_bronze(raw_data)

    print("[INFO] Saving bronze JSON...")
    save_bronze_json(bronze_data)


if __name__ == "__main__":
    run_pipeline()

[INFO] Loading raw data...
[INFO] Transforming to bronze layer...
[INFO] Saving bronze JSON...
[SUCCESS] Bronze data saved to fng_bronze_20260401_104658.json


C:\Users\Christopher\AppData\Local\Temp\ipykernel_28264\1331052837.py:24: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "date": datetime.utcfromtimestamp(int(row["timestamp"])).strftime("%Y-%m-%d"),
